# Analyses inférentielles (Test statistiques)

Analyses retenues

. Welch (moyennes) et Mann–Whitney (rangs)  : quantitatives vs classe

. Test exact de Fisher                      : qualitatives vs classe, proportions de patients exposés

. Facteur d'inflation de la variance (VIF)  : multicolinéarité

. Régression logistique multiple            : ajustement tenté, séparation parfaite constatée

. Sphéricité de Bartlett                    : préalable à l'ACP

In [1]:
# IMPORTATIONS

from pathlib import Path

import numpy as np

import pandas as pd

from scipy import stats

from statsmodels.stats.multitest import multipletests

import statsmodels.api as sm

from statsmodels.stats.outliers_influence import variance_inflation_factor

In [2]:
import warnings

warnings.filterwarnings("ignore")

In [3]:
# CONFIGURATIONS

DATA = Path("..") / "data" / "kidney_clean.csv"
SEUIL_VIF = 10          # seuil usuel de multicolinéarité
SEUIL_MANQUANTS = 0.20  # au-delà, variable exclue du modèle multiple

VARS_NUM = ["age", "bp", "sg", "al", "su", "bgr", "bu", "sc", "sod", "pot", "hemo", "pcv", "wc", "rc"]

VARS_CAT = {"htn": "yes", "dm": "yes", "cad": "yes", "appet": "poor", "pe": "yes",
       "ane": "yes", "pcc": "present", "ba": "present", "pc": "abnormal", "rbc": "abnormal"}

BLOCS = {"hématologie": ["hemo", "pcv", "rc", "wc"], 
         "électrolytes": ["sod", "pot"],
         "rénal": ["sc", "bu"], 
         "urinaire": ["sg", "al", "su"]}

# 1. Chargement des données

In [4]:
fichier = Path("../data/kidney_clean.csv")
if not fichier.exists():
    raise FileNotFoundError(
        "Fichier introuvable : placez kidney_clean.csv dans le même dossier que ce notebook."
    )

df = pd.read_csv(fichier)
df.columns = df.columns.str.strip().str.lower()

if "classification" not in df.columns:
    raise ValueError("La colonne 'classification' est absente du fichier.")

df["classification"] = df["classification"].astype(str).str.strip().str.lower()
df = df[df["classification"].isin(["ckd", "notckd"])].copy()
df["ckd"] = (df["classification"] == "ckd").astype(int)

print(f"Nombre d'observations : {len(df)}")
print(f"Patients ckd          : {(df['ckd'] == 1).sum()}")
print(f"Patients notckd       : {(df['ckd'] == 0).sum()}")
display(df.head())

Nombre d'observations : 400
Patients ckd          : 250
Patients notckd       : 150


,id,age,bp,sg,al,su,rbc,pc,pcc,ba,...,wc,rc,htn,dm,cad,appet,pe,ane,classification,ckd
0,0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,...,7800.0,5.2,yes,yes,no,good,no,no,ckd,1
1,1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,...,6000.0,NaN,no,no,no,good,no,no,ckd,1
2,2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,...,7500.0,NaN,no,yes,no,poor,no,yes,ckd,1
3,3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,...,6700.0,3.9,yes,no,no,poor,yes,yes,ckd,1
4,4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,...,7300.0,4.6,no,no,no,good,no,no,ckd,1


# 2. Test de Welch et Test de Mann-Whitney

**Question :** les variables quantitatives diffèrent-elles entre les groupes `ckd` et `notckd` ?

- **Welch** teste si les moyennes des groupes ckd et notckd sont différentes, sans supposer que leurs variances sont égales. Il est donc plus robuste que le test t classique lorsque les dispersions ou les effectifs diffèrent.
- **Mann–Whitney** compare la position globale des observations (rangs) entre les deux groupes. Il détecte une différence de distribution et n'est interprétable comme un test des médianes que lorsque les distributions ont une forme similaire.
- Comme plusieurs variables sont analysées simultanément, la correction de Benjamini–Hochberg (p_BH) contrôle le taux de fausses découvertes et réduit le risque d'identifier à tort des différences significatives.

In [5]:
variables_num = [
    v for v in ["age", "bp", "sg", "al", "su", "bgr", "bu", "sc",
                "sod", "pot", "hemo", "pcv", "wc", "rc"]
    if v in df.columns
]

for v in variables_num:
    df[v] = pd.to_numeric(df[v], errors="coerce")

resultats_welch = []

for v in variables_num:
    x = df.loc[df["ckd"] == 1, v].dropna()
    y = df.loc[df["ckd"] == 0, v].dropna()

    if len(x) < 2 or len(y) < 2:
        continue

    test_w = stats.ttest_ind(x, y, equal_var=False)
    test_mw = stats.mannwhitneyu(x, y, alternative="two-sided")

    diff = x.mean() - y.mean()
    se = np.sqrt(x.var(ddof=1) / len(x) + y.var(ddof=1) / len(y))
    ddl = (x.var(ddof=1) / len(x) + y.var(ddof=1) / len(y))**2 / (
        (x.var(ddof=1) / len(x))**2 / (len(x)-1)
        + (y.var(ddof=1) / len(y))**2 / (len(y)-1)
    )
    marge = stats.t.ppf(0.975, ddl) * se

    ecart_type_combine = np.sqrt(
        ((len(x)-1)*x.var(ddof=1) + (len(y)-1)*y.var(ddof=1))
        / (len(x)+len(y)-2)
    )
    d_cohen = diff / ecart_type_combine if ecart_type_combine > 0 else np.nan

    resultats_welch.append({
        "variable": v, "n_ckd": len(x), "n_notckd": len(y),
        "moy_ckd": x.mean(), "moy_notckd": y.mean(),
        "med_ckd": x.median(), "med_notckd": y.median(),
        "diff_moyennes": diff, "IC95_inf": diff-marge, "IC95_sup": diff+marge,
        "Cohen_d": d_cohen, "p_Welch": test_w.pvalue, "p_Mann_Whitney": test_mw.pvalue
    })

tab_quant = pd.DataFrame(resultats_welch)
if not tab_quant.empty:
    from statsmodels.stats.multitest import multipletests
    tab_quant["p_Welch_BH"] = multipletests(tab_quant["p_Welch"], method="fdr_bh")[1]
    tab_quant["p_MW_BH"] = multipletests(tab_quant["p_Mann_Whitney"], method="fdr_bh")[1]
    display(tab_quant.round(4))
else:
    print("Aucune variable quantitative ne dispose d'assez de valeurs dans les deux groupes.")

,variable,n_ckd,n_notckd,moy_ckd,moy_notckd,med_ckd,med_notckd,diff_moyennes,IC95_inf,IC95_sup,Cohen_d,p_Welch,p_Mann_Whitney,p_Welch_BH,p_MW_BH
0,age,242,149,54.5413,46.5168,59.000,46.00,8.0245,4.6811,11.3680,0.4793,0.0000,0.0000,0.0000,0.0000
1,bp,240,148,79.6250,71.3514,80.000,70.00,8.2736,5.8978,10.6495,0.6318,0.0000,0.0000,0.0000,0.0000
2,sg,208,145,1.0139,1.0224,1.015,1.02,-0.0085,-0.0092,-0.0077,-2.1787,0.0000,0.0000,0.0000,0.0000
3,al,209,145,1.7225,0.0000,2.000,0.00,1.7225,1.5353,1.9097,1.6324,0.0000,0.0000,0.0000,0.0000
4,su,206,145,0.7670,0.0000,0.000,0.00,0.7670,0.5817,0.9522,0.7421,0.0000,0.0000,0.0000,0.0000
5,bgr,212,144,175.4198,107.7222,143.500,107.50,67.6976,54.8710,80.5242,0.9394,0.0000,0.0000,0.0000,0.0000
6,bu,237,144,72.3890,32.7986,53.000,33.00,39.5904,31.8653,47.3155,0.8466,0.0000,0.0000,0.0000,0.0000
7,sc,238,145,4.4149,0.8690,2.250,0.90,3.5460,2.6575,4.4344,0.6466,0.0000,0.0000,0.0000,0.0000
8,sod,167,145,134.6766,141.7310,136.000,141.00,-7.0544,-8.4167,-5.6920,-1.1251,0.0000,0.0000,0.0000,0.0000
9,pot,165,145,4.4164,4.3379,4.300,4.50,0.0784,-0.0868,0.2437,0.1037,0.3511,0.9306,0.3511,0.9306


### Interprétation

Après correction de Benjamini-Hochberg, 13 des 14 variables quantitatives diffèrent significativement entre les groupes ckd et notckd (p_BH < 0,05).. Les patients `ckd` présentent notamment des valeurs plus élevées de pression artérielle, d'urée, de créatinine et de glycémie, ainsi que des valeurs plus faibles de densité urinaire spécifique, d'hémoglobine, d'hématocrite et de globules rouges.

Les plus fortes différences concernent l'hémoglobine, l'hématocrite, les globules rouges et la densité urinaire spécifique (tailles d'effet très importantes). Seul le potassium ne montre pas de différence significative entre les deux groupes (`p > 0,05`).

# 3. Test exact de Fisher et odds ratios

**Question :** existe-t-il une association entre une variable qualitative et la classe `ckd` ?

Le test exact de Fisher évalue l'existence d'une association entre deux variables qualitatives dans un tableau 2×2, même lorsque certains effectifs sont faibles.

L'odds ratio (OR) mesure la force de cette association : un OR > 1 indique que la modalité est plus fréquente chez les patients `ckd`, un OR < 1 qu'elle est moins fréquente, et un OR = 1 l'absence d'association.
``

In [10]:
variables_cat = {
    "htn": "yes", "dm": "yes", "cad": "yes", "appet": "poor",
    "pe": "yes", "ane": "yes", "pcc": "present", "ba": "present",
    "pc": "abnormal", "rbc": "abnormal"
}
resultats_fisher = []

for v, modalite in variables_cat.items():
    if v not in df.columns:
        continue

    s = df[[v, "ckd"]].dropna().copy()
    s[v] = s[v].astype(str).str.strip().str.lower()
    expose = s[v] == modalite

    a = int(((expose) & (s["ckd"] == 1)).sum())
    b = int(((expose) & (s["ckd"] == 0)).sum())
    c = int(((~expose) & (s["ckd"] == 1)).sum())
    d = int(((~expose) & (s["ckd"] == 0)).sum())

    p_fisher = stats.fisher_exact([[a, b], [c, d]], alternative="two-sided").pvalue

    # Si une case du tableau 2x2 est nulle, l'OR n'est pas défini : on ne l'estime pas.
    if 0 in [a, b, c, d]:
        or_, ic_inf, ic_sup = np.nan, np.nan, np.nan
    else:
        or_ = (a * d) / (b * c)
        se_log_or = np.sqrt(1/a + 1/b + 1/c + 1/d)
        ic_inf = np.exp(np.log(or_) - 1.96 * se_log_or)
        ic_sup = np.exp(np.log(or_) + 1.96 * se_log_or)

    resultats_fisher.append({
        "variable": v, "modalite_etudiee": modalite,
        "ckd_expose": a, "notckd_expose": b,
        "ckd_non_expose": c, "notckd_non_expose": d,        
        "prop_ckd": a/(a+c), "prop_notckd": b/(b+d),
        "OR": or_, "IC95_inf": ic_inf, "IC95_sup": ic_sup, "p_Fisher": p_fisher
    })

tab_fisher = pd.DataFrame(resultats_fisher)
if not tab_fisher.empty:
    tab_fisher["p_Fisher_BH"] = multipletests(tab_fisher["p_Fisher"], method="fdr_bh")[1]
    display(tab_fisher.round(4))
else:
    print("Aucune variable qualitative attendue n'a été trouvée.")

,variable,modalite_etudiee,ckd_expose,notckd_expose,ckd_non_expose,notckd_non_expose,prop_ckd,prop_notckd,OR,IC95_inf,IC95_sup,p_Fisher,p_Fisher_BH
0,htn,yes,147,0,103,148,0.5880,0.0,NaN,NaN,NaN,0.0,0.0
1,dm,yes,137,0,113,148,0.5480,0.0,NaN,NaN,NaN,0.0,0.0
2,cad,yes,34,0,216,148,0.1360,0.0,NaN,NaN,NaN,0.0,0.0
3,appet,poor,82,0,168,149,0.3280,0.0,NaN,NaN,NaN,0.0,0.0
4,pe,yes,76,0,174,149,0.3040,0.0,NaN,NaN,NaN,0.0,0.0
5,ane,yes,60,0,190,149,0.2400,0.0,NaN,NaN,NaN,0.0,0.0
6,pcc,present,42,0,208,146,0.1680,0.0,NaN,NaN,NaN,0.0,0.0
7,ba,present,22,0,228,146,0.0880,0.0,NaN,NaN,NaN,0.0,0.0
8,pc,abnormal,76,0,118,141,0.3918,0.0,NaN,NaN,NaN,0.0,0.0
9,rbc,abnormal,47,0,60,141,0.4393,0.0,NaN,NaN,NaN,0.0,0.0


### Interprétation

Les 10 variables qualitatives étudiées sont significativement associées à la classe `ckd` après correction de Benjamini-Hochberg (`p_BH < 0,001` pour toutes).

Pour chacune, **aucun patient `notckd` ne présente la modalité étudiée**, alors que 8,8 % (`ba`) à 58,8 % (`htn`) des patients `ckd` la présentent. Par exemple, 147 patients `ckd` sur 250 (58,8 %) sont hypertendus, contre 0 sur 148 patients `notckd` ; 137 sur 250 (54,8 %) sont diabétiques, contre 0 sur 148. Les proportions sont calculées sur les patients dont la variable est renseignée.

Une case du tableau 2×2 étant nulle, l'odds ratio n'est pas défini : il n'est donc pas estimé, et il ne permet pas de classer les variables entre elles. La mesure à retenir est la proportion de patients exposés dans chaque groupe.

Ces modalités (`yes`, `poor`, `present`, `abnormal`) sont donc observées uniquement chez des patients `ckd`. Leur présence est associée à la maladie, mais leur absence n'est pas informative : par exemple, 103 patients `ckd` sur 250 ne sont pas hypertendus.

# 4. Multicolinéarité : facteur d'inflation de la variance (VIF)

Avant une régression multiple, le VIF permet de détecter les variables explicatives fortement corrélées entre elles. Une valeur élevée indique une redondance d'information pouvant rendre les coefficients du modèle instables et difficiles à interpréter.

Cette analyse porte sur les 10 variables quantitatives ayant au plus 20 % de valeurs manquantes, pour les 260 patients dont les mesures sont complètes.

In [7]:
seuil_manquants = 0.20
variables_vif = [
    v for v in variables_num
    if df[v].isna().mean() <= seuil_manquants
]

X_vif = df[variables_vif].dropna()
if len(variables_vif) >= 2 and len(X_vif) > len(variables_vif) + 1:
    X_vif = X_vif.loc[:, X_vif.nunique() > 1]
    X_std = (X_vif - X_vif.mean()) / X_vif.std(ddof=0)
    tableau_vif = pd.DataFrame({
        "variable": X_std.columns,
        "VIF": [variance_inflation_factor(X_std.to_numpy(), i)
                for i in range(X_std.shape[1])]
    }).sort_values("VIF", ascending=False)
    display(tableau_vif.round(3))
else:
    print("Pas assez de données complètes pour calculer les VIF.")

,variable,VIF
8,hemo,5.152
9,pcv,4.710
5,bgr,2.745
6,bu,2.627
4,su,2.520
7,sc,2.437
3,al,2.293
2,sg,2.136
1,bp,1.161
0,age,1.157


### Interprétation

Les valeurs de VIF sont toutes inférieures à 10, ce qui indique l'absence de multicolinéarité sévère entre les variables explicatives. Les variables `hemo` (VIF = 5,15) et `pcv` (VIF = 4,71) présentent les niveaux de colinéarité les plus élevés, mais restent dans une plage généralement acceptable.

Globalement, aucune variable ne semble suffisamment redondante pour justifier son exclusion sur la seule base du VIF.

# 5. Régression logistique multiple

**Question :** parmi toutes les variables quantitatives étudiées, lesquelles restent associées à `ckd` lorsque l'on les considère simultanément ?

Le modèle porte sur les 10 variables quantitatives ayant au plus 20 % de valeurs manquantes (sod, pot, wc et rc sont exclues), standardisées, et sur les 260 patients dont les mesures sont complètes.

Une variable significative isolément peut cesser de l’être ici si son effet est expliqué par les autres.

In [11]:
variables_modele = [
    v for v in variables_num
    if df[v].isna().mean() <= seuil_manquants and df[v].nunique(dropna=True) > 1
]
donnees_modele = df[variables_modele + ["ckd"]].dropna().copy()

print("Variables candidates :", variables_modele)
print("Observations complètes :", len(donnees_modele))
print("CKD :", int(donnees_modele["ckd"].sum()),
      "| notCKD :", int((donnees_modele["ckd"] == 0).sum()))

if len(variables_modele) == 0 or len(donnees_modele) <= len(variables_modele) + 5:
    print("Attention : données insuffisantes pour ajuster sereinement le modèle multiple.")
else:
    X = donnees_modele[variables_modele].astype(float)
    X = (X - X.mean()) / X.std(ddof=0)
    X = sm.add_constant(X)
    y = donnees_modele["ckd"].astype(int)

    try:
        with warnings.catch_warnings(record=True) as avertissements:
            warnings.simplefilter("always")
            modele = sm.Logit(y, X).fit(disp=False, maxiter=200)

        print("\nConvergence atteinte :", modele.mle_retvals["converged"])
        for msg in sorted({f"{a.category.__name__} : {a.message}" for a in avertissements}):
            print("Avertissement :", msg)
        print("\nCoefficients (variables standardisées) :")
        print(modele.params.round(2).to_string())
        probas = modele.predict(X)
        print("\nPatients classés correctement :", int(((probas > 0.5) == (y == 1)).sum()), "sur", len(y))
    except Exception as erreur:
        modele = None
        print("Le modèle logistique standard n'a pas pu être estimé :", erreur)
        print("Vérifiez la colinéarité, la séparation et le nombre d'observations.")

Variables candidates : ['age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc', 'hemo', 'pcv']
Observations complètes : 260
CKD : 130 | notCKD : 130

Convergence atteinte : False
Avertissement : ConvergenceWarning : Maximum Likelihood optimization failed to converge. Check mle_retvals
Avertissement : PerfectSeparationWarning : Perfect separation or prediction detected, parameter may not be identified

Coefficients (variables standardisées) :
const    105.65
age      -19.09
bp        -3.58
sg       -45.87
al       125.05
su       -14.30
bgr       68.72
bu        -0.17
sc        15.95
hemo     -53.89
pcv      -26.25

Patients classés correctement : 260 sur 260


### Interprétation

Le modèle logistique standard **ne converge pas** (`Convergence atteinte : False`) et signale une **séparation parfaite** : avec les 10 variables retenues, les 260 patients sont tous classés correctement (260 sur 260). Les coefficients obtenus (par exemple 125 pour `al`, −54 pour `hemo`) n'ont aucune signification statistique : ils ne doivent pas être interprétés, ni transformés en odds ratios.

Cette séparation vient de la **combinaison** des variables, pas d'une variable seule : aucune variable ne sépare à elle seule les deux groupes, leurs distributions se chevauchent. C'est une propriété du jeu de données, pas une erreur de calcul.

Le modèle n'utilise que 260 patients sur 400 : les variables ayant plus de 20 % de valeurs manquantes (`sod`, `pot`, `wc`, `rc`) sont exclues, puis les patients incomplets. Cela retire 120 patients `ckd` sur 250 (48 %) et 20 patients `notckd` sur 150 (13 %), d'où des groupes équilibrés de 130 patients chacun, mais non représentatifs de l'ensemble.

# 6. Odds ratios du modèle multiple

**Question :** quelle est l'influence de chaque variable sur la probabilité d'appartenir au groupe `ckd`, après ajustement sur les autres variables ?

Les odds ratios (OR) du modèle multiple permettent normalement de quantifier l'effet propre de chaque variable. Toutefois, le modèle n'ayant pas convergé (séparation parfaite, section 5), les OR et leurs intervalles de confiance ne sont **pas calculés** : ils seraient instables et trompeurs.

Pour comparer les variables entre elles, on se reporte aux tailles d'effet de la section 2 (d de Cohen), où `hemo`, `pcv`, `sg` et `rc` présentent les différences les plus marquées.





# 7. Test de sphéricité de Bartlett — préalable à l'ACP

**Question :** les variables sont-elles suffisamment corrélées pour réaliser une ACP ?

Le test de Bartlett vérifie si la matrice de corrélation contient suffisamment de relations entre les variables pour justifier une analyse en composantes principales (ACP).

- **H₀** : les variables ne sont pas corrélées (matrice de corrélation identité).
- **H₁** : au moins certaines variables sont corrélées.

In [9]:
variables_acp = [
    v for v in variables_num
    if df[v].isna().mean() <= seuil_manquants and df[v].nunique(dropna=True) > 1
]
X_acp = df[variables_acp].dropna()

n, p = X_acp.shape
if p < 2 or n <= p:
    print("Bartlett non calculé : il faut au moins 2 variables et suffisamment d'observations complètes.")
else:
    R = np.corrcoef(X_acp.to_numpy(), rowvar=False)
    signe, logdet = np.linalg.slogdet(R)

    if signe <= 0 or not np.isfinite(logdet):
        print("Bartlett non calculé : la matrice de corrélation est singulière ou non exploitable.")
    else:
        chi2_b = -(n - 1 - (2*p + 5)/6) * logdet
        ddl_b = p*(p-1)//2
        p_b = stats.chi2.sf(chi2_b, ddl_b)

        print(f"Observations complètes n = {n}")
        print(f"Variables p = {p}")
        print(f"Chi-deux = {chi2_b:.3f}")
        print(f"Degrés de liberté = {ddl_b}")
        print(f"p-value = {p_b:.4g}")

Observations complètes n = 260
Variables p = 10
Chi-deux = 1447.566
Degrés de liberté = 45
p-value = 6.063e-274


### Interprétation
Le test de Bartlett est hautement significatif (χ² = 1447,57 ; ddl = 45 ; p < 0,001). Calculé sur les 10 variables retenues et les 260 patients complets, il rejette l’hypothèse d’absence de corrélation : les variables sont suffisamment corrélées pour envisager une ACP.

Ce résultat n’est pas comparable à celui du notebook 04 (analyse multivariée), qui porte sur 14 variables du fichier imputé.

## Conclusion générale

Les analyses univariées montrent de fortes différences entre les groupes ckd et notckd : 13 des 14 variables quantitatives (seul pot fait exception) et les 10 variables qualitatives étudiées sont significativement associées à la maladie après correction de Benjamini–Hochberg. Les tailles d’effet les plus fortes concernent hemo, pcv, sg et rc (|d| de Cohen entre 1,95 et 2,43). Pour les variables qualitatives, aucun patient notckd ne présente les modalités étudiées (yes, poor, present, abnormal) : leur présence est associée à la maladie, mais leur absence n’est pas informative.

La régression logistique multiple n’a pas pu être estimée de façon exploitable : la séparation est parfaite (260 patients sur 260 classés correctement) et les coefficients ne sont pas interprétables. Cette séparation vient de la combinaison des variables, aucune ne séparant les groupes à elle seule. Le modèle repose en outre sur 260 patients sur 400, non représentatifs (48 % des ckd écartés contre 13 % des notckd). Il ne permet donc pas d’identifier de facteurs déterminants ; l’importance relative des variables est appréciée par les tailles d’effet de la section 2.

Enfin, le VIF ne révèle pas de multicolinéarité sévère (maximum 5,15, pour hemo), et le test de Bartlett (χ² = 1447,57 ; ddl = 45 ; p < 0,001), calculé sur 10 variables et 260 patients complets, justifie le recours à une analyse en composantes principales.